# 24 -- Roadmap item 6: 2D thick-slab CNN, architecture diversity (2026-09-11)

The shipped 6-variant/150-checkpoint ensemble is calibration-exhausted
(+0.0020 excess log loss vs. its own AUROC's theoretical optimum -- Opus
review, 2026-09-11). Three cheap recombination candidates tried today
(`notebooks/23_paired_gate_shipped_recipe_candidates.ipynb`) all came back
null, confirming that nothing which only reweights/recombines the *same*
3D-CNN ensemble's outputs can move the score further -- every existing
variant differs only by training protocol (LR schedule, augmentation,
class weight, family oversampling, fixed-epoch training), never by
architecture. **Roadmap item 6** (`README.md`, open since 2026-09-10:
"bounded architecture check -- a 2D-slab CNN or similar as an additional
ensemble member -- never actually run") is the only remaining lever with
an actual discrimination (AUROC) mechanism, not just recalibration.

**Architecture**: `model.DatSlab2DCNN` (`src/model.py`) -- a 2D CNN over a
thick axial "slab" through the striatum, following **Wenzel et al. 2019**
(EJNMMI, `RESOURCES.md`): 2x2x12mm voxels (high in-plane resolution, one
12mm-thick slice along the S-I axis, not a full 3D volume), 4 conv blocks
(64/64/96/128, 3x3, batchnorm, 2x2 maxpool) -> dense head. This is a
genuinely different inductive bias from the existing `DatCNN` (full 3D
volume, 1->16->32->64->128 channels), so its errors should be decorrelated
from the existing ensemble's for reasons other than training-protocol
noise.

**Data**: `data.load_slab` (`src/data.py`) reuses the exact same
`resample_to_spacing`/`crop_or_pad`/`normalize_intensity` building blocks
as `load_volume`, resampled to `config.SLAB_TARGET_SPACING = (2.0, 2.0,
2.0)` mm and cropped to `config.SLAB_TARGET_SHAPE = (68, 35, 6)` voxels
(6x2mm = 12mm total S-I thickness, averaged into one 2D image) at the same
striatum-centered `config.CROP_CENTER_MM` offset already validated for the
3D track. **Revised 2026-09-11** from an initial single-12mm-voxel design
after the fold-0 sanity check scored barely above the base-rate baseline
-- a synthetic-marker diagnostic confirmed `crop_or_pad`'s nearest-voxel
rounding (+-0.5 voxel = +-6mm at 12mm spacing) is negligible for the 3D
track's 108mm-thick crop but large enough to miss the striatum entirely
at 1x12mm; averaging 6 finer (2mm) voxels keeps the same physical
thickness while cutting that error to +-1mm (`config.py`'s comment above
`SLAB_TARGET_SPACING` has the full diagnosis). **If you already ran this
notebook with the earlier config, the on-disk `slab2d_cache` will detect
the changed fingerprint and rebuild automatically -- no manual cleanup
needed, but expect the ~13min cache-build cost again.**

**Training protocol**: deliberately the *simplest* possible baseline for
this new architecture -- plain `BCEWithLogitsLoss`, Adam,
`batch_size=32, lr=2e-3, weight_decay=1e-5` (this project's own established
defaults, rung-3's original recipe), standard nested CV with an inner-
validation split driving early stopping (`evaluate.make_folds(...,
n_splits=10)[0]`, same pattern as `notebooks/09`) -- **no augmentation, no
class-weighting, no LR schedule**. Those are separate, already-explored
axes on the 3D architecture; stacking them onto a brand-new architecture in
the same experiment would confound "does a different architecture help the
ensemble" with "does augmentation help this architecture." 5 seeds x 5
folds = 25 fold-trainings, prefix `slab2d`.

**Gate**: `evaluate.oof_predict`/`evaluate.paired_gate` (promoted today
from notebook 23's corrected paired-bootstrap method -- NOT notebook 22's
buggy `op06denoise`-style unpaired-fold-sd comparison). Decision rule:
whole 95% CI negative AND `|delta| > 0.003`, same bar notebook 23 already
established.

**Data handling**: loads real row-level labels and, eventually, real
`.nii.gz` volumes via the injected `data.load_slab` -- per the AI-assistant
data rule (`README.md`), this is **[RUN ME]**: run it yourself, share back
only printed aggregate metrics (never per-row/per-uid output). Run the
fold-0 sanity-check cell first (cheap proxy, catches shape/convergence bugs
before committing to the full 25-fold run) before the full nested-CV loop.

In [ ]:
# [RUN ME] -- loads real row-level labels; builds a NEW, separate on-disk
# cache for slab2d's own preprocessing (data.load_slab), distinct from the
# existing 3D volume_cache -- different spacing/shape, must not overwrite
# the 3D track's cache.
import sys
import time
import warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import torch

import cache
import config
import data
import dataset
import evaluate
import model
import submission
import train as train_mod

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)

uids = labeled_df[config.UID_COLUMN].tolist()
labels = labeled_df[config.TARGET_COLUMN].tolist()
families = labeled_df["inplane_family"].tolist()

slab_config_fingerprint = {
    "SLAB_TARGET_SPACING": config.SLAB_TARGET_SPACING,
    "SLAB_TARGET_SHAPE": config.SLAB_TARGET_SHAPE,
    "CROP_CENTER_MM": config.CROP_CENTER_MM,
    "BACKGROUND_PERCENTILE": config.BACKGROUND_PERCENTILE,
    "BACKGROUND_MAX_FRACTION": config.BACKGROUND_MAX_FRACTION,
}

# Python's default warning filter shows a UserWarning only once per unique
# (message, source line) -- with a shared message and a single call site
# (data.py::_warn_if_degenerate), that would silently hide how many uids
# actually triggered it. simplefilter("always") + record=True gets an
# honest per-uid count instead of a possibly-misleading single printout,
# so a rare/expected handful of geometric outliers (this project's own
# precedent -- config.py documents a few tight-FOV/atypical volumes for
# the 3D track) isn't confused with a systemic slab-geometry bug.
cache_start = time.time()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    slab_cache = cache.CachedVolumeStore(
        uids, cache_dir=config.DATA_PROCESSED / "slab2d_cache",
        config_fingerprint=slab_config_fingerprint, load_fn=data.load_slab,
    )
degenerate_count = sum(1 for w in caught if "degenerate/near-empty volume" in str(w.message))
print(f"slab cache {'reused' if slab_cache.was_reused else 'rebuilt'} in "
      f"{time.time() - cache_start:.1f}s for {len(uids)} volumes")
print(f"degenerate/near-empty slab warning fired for {degenerate_count}/{len(uids)} volumes "
      f"({degenerate_count / len(uids):.1%})")
if degenerate_count / len(uids) > 0.01:
    print("^ that's >1% of volumes -- worth investigating the slab geometry "
          "(config.SLAB_TARGET_SPACING/SHAPE, CROP_CENTER_MM) before trusting "
          "the training run below, not just noise to ignore.")

In [ ]:
# [RUN ME] (no new data access -- reuses the slab_cache built above).
# Same nested-CV trainer shape as notebooks/09/07's train_and_score_nested,
# with no oversampling/class-weighting (out of scope for this architecture-
# diversity check -- see intro cell) and model.build_slab_model() instead
# of model.build_model().
def train_and_score_slab(train_uids, train_labels, train_family, outer_uids,
                          batch_size, lr, seed, epochs=config.EPOCHS,
                          patience=config.PATIENCE, inner_splits=10):
    inner_train_idx, inner_val_idx = evaluate.make_folds(
        train_labels, train_family, n_splits=inner_splits, random_state=seed
    )[0]

    def subset(idxs):
        return [train_uids[i] for i in idxs], [train_labels[i] for i in idxs]

    inner_train_uids, inner_train_labels = subset(inner_train_idx)
    inner_val_uids, inner_val_labels = subset(inner_val_idx)

    inner_train_ds = dataset.DatParkinsonDataset(inner_train_uids, inner_train_labels, load_fn=slab_cache.get)
    inner_val_ds = dataset.DatParkinsonDataset(inner_val_uids, inner_val_labels, load_fn=slab_cache.get)
    inner_train_loader = torch.utils.data.DataLoader(inner_train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    inner_val_loader = torch.utils.data.DataLoader(inner_val_ds, batch_size=batch_size, num_workers=0)

    torch.manual_seed(seed)
    net = model.build_slab_model().to(config.DEVICE)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=config.WEIGHT_DECAY)
    loss_fn = torch.nn.BCEWithLogitsLoss()

    best_state, history = train_mod.train_one_fold(
        net, inner_train_loader, inner_val_loader, optimizer, loss_fn,
        epochs=epochs, patience=patience, device=config.DEVICE,
        use_amp=config.USE_AMP, seed=seed,
    )
    net.load_state_dict(best_state)

    outer_ds = dataset.DatParkinsonDataset(outer_uids, load_fn=slab_cache.get)
    outer_loader = torch.utils.data.DataLoader(outer_ds, batch_size=batch_size, num_workers=0)
    outer_probs = []
    for x, _ in outer_loader:
        outer_probs.append(model.predict(net, x))
    return np.concatenate(outer_probs), history, best_state


print("train_and_score_slab defined.")

In [ ]:
# [RUN ME] -- cheap proxy (deep-learning-imaging.md's cheap-proxy ladder):
# fold-0 only, before committing to the full 25-fold run. Catches shape/
# convergence bugs cheaply. Does NOT decide anything on its own -- only
# the full nested-CV + gate below can adopt or reject this variant.
batch_size, lr = 32, 2e-3

outer_folds = evaluate.make_folds(np.array(labels), np.array(families),
                                   n_splits=config.N_FOLDS, random_state=config.SEED)
fold0_train_idx, fold0_test_idx = outer_folds[0]
fold0_train_uids = [uids[i] for i in fold0_train_idx]
fold0_train_labels = [labels[i] for i in fold0_train_idx]
fold0_train_family = [families[i] for i in fold0_train_idx]
fold0_test_uids = [uids[i] for i in fold0_test_idx]
fold0_test_labels = np.array([labels[i] for i in fold0_test_idx])

start = time.time()
probs, history, best_state = train_and_score_slab(
    fold0_train_uids, fold0_train_labels, fold0_train_family,
    fold0_test_uids, batch_size=batch_size, lr=lr, seed=config.SEED,
)
elapsed = time.time() - start
score = evaluate.log_loss_score(fold0_test_labels, probs)
print(f"fold-0 sanity check: {len(history['val_loss'])} epochs, "
      f"inner-val best={min(history['val_loss']):.4f}, outer fold log loss={score:.4f}, "
      f"{elapsed:.1f}s ({elapsed / len(history['val_loss']):.2f}s/epoch)")
print(f"inner-val loss history: {[round(v, 4) for v in history['val_loss']]}")
print("expect: loss visibly decreasing over the run, not flat/NaN/exploding "
      "-- if it looks wrong, stop here and debug before the full 25-fold run below.")

In [ ]:
# [RUN ME] -- full 5-fold nested CV, repeated 5x (25 fold-trainings total).
# Only run this after the sanity check above looks reasonable.
N_REPEATS = 5
oof_repeats_slab2d = []

for repeat_seed in range(config.SEED, config.SEED + N_REPEATS):
    outer_folds = evaluate.make_folds(np.array(labels), np.array(families),
                                       n_splits=config.N_FOLDS, random_state=repeat_seed)
    oof_probs = np.zeros(len(uids))
    for fold_i, (train_idx, test_idx) in enumerate(outer_folds):
        fold_train_uids = [uids[i] for i in train_idx]
        fold_train_labels = [labels[i] for i in train_idx]
        fold_train_family = [families[i] for i in train_idx]
        fold_test_uids = [uids[i] for i in test_idx]

        start = time.time()
        probs, history, best_state = train_and_score_slab(
            fold_train_uids, fold_train_labels, fold_train_family,
            fold_test_uids, batch_size=batch_size, lr=lr, seed=repeat_seed,
        )
        elapsed = time.time() - start
        oof_probs[test_idx] = probs
        torch.save(best_state, config.CHECKPOINT_DIR / f"slab2d_seed{repeat_seed}_fold{fold_i}.pt")
        fold_score = evaluate.log_loss_score(np.array(labels)[test_idx], probs)
        print(f"  seed={repeat_seed} fold={fold_i}: {len(history['val_loss'])} epochs, "
              f"outer fold log loss={fold_score:.4f}, {elapsed:.1f}s")

    repeat_logloss = evaluate.log_loss_score(np.array(labels), oof_probs)
    oof_repeats_slab2d.append(oof_probs)
    print(f"seed={repeat_seed} pooled OOF log loss: {repeat_logloss:.4f}")
    np.save(config.DATA_PROCESSED / f"slab2d_oof_seed{repeat_seed}.npy", oof_probs)

repeat_scores_slab2d = np.array([evaluate.log_loss_score(np.array(labels), oof) for oof in oof_repeats_slab2d])
print(f"\n{N_REPEATS}-repeat slab2d CNN pooled log loss: "
      f"mean={repeat_scores_slab2d.mean():.4f}, sd={repeat_scores_slab2d.std(ddof=1):.4f}")
print("current validated 3D CNN alone (rung 3, README.md 2026-09-09): mean=0.4520, sd=0.0109")
print("(slab2d is NOT expected to beat the 3D CNN alone -- what matters is "
      "whether it adds value as a 7th ENSEMBLE member, gated below.)")

In [ ]:
# [RUN ME] -- the real decision: does slab2d add value as a 7th ensemble
# member, using the CORRECTED paired-bootstrap gate (evaluate.oof_predict/
# evaluate.paired_gate, promoted from notebook 23 today) -- NOT notebook
# 22's buggy op06denoise-style unpaired-fold-sd comparison.
family_arr = np.array(families)
y_true = np.array(labels)
repeat_seeds = list(range(config.SEED, config.SEED + 5))

cnn_arrays = [
    np.load(config.DATA_PROCESSED / f"{prefix}_oof_seed{s}.npy")
    for prefix in model.PRODUCTION_VARIANT_PREFIXES for s in repeat_seeds
]
slab2d_arrays = [
    np.load(config.DATA_PROCESSED / f"slab2d_oof_seed{s}.npy") for s in repeat_seeds
]
baseline_oof_repeats = [np.load(config.DATA_PROCESSED / f"baseline_oof_seed{s}.npy") for s in repeat_seeds]
baseline_pooled = np.mean(baseline_oof_repeats, axis=0)

FOLDS = evaluate.make_folds(y_true, family_arr, n_splits=config.N_FOLDS, random_state=config.RANDOM_STATE)


def blend_features(cnn_p, baseline_p):
    return np.column_stack([submission.to_logit(cnn_p), submission.to_logit(baseline_p)])


cnn_pooled_6variant = submission.pool_logit_mean(cnn_arrays)
current_oof = evaluate.oof_predict(blend_features(cnn_pooled_6variant, baseline_pooled), y_true, FOLDS)

cnn_pooled_7variant = submission.pool_logit_mean(cnn_arrays + slab2d_arrays)
candidate_oof = evaluate.oof_predict(blend_features(cnn_pooled_7variant, baseline_pooled), y_true, FOLDS)

result = evaluate.paired_gate("6-variant vs. 7-variant(+slab2d)", candidate_oof, current_oof, y_true,
                               min_effect=0.003, seed=config.RANDOM_STATE)
print("for comparison -- shipped 6-variant recipe's own honest row-wise CV "
      "(notebooks 22/23): log loss=0.3617, AUROC=0.9184")
print(f"7-variant(+slab2d) row-wise CV AUROC: {evaluate.auroc_score(y_true, candidate_oof):.4f}")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays built in op06gate above).
# A second Opus review (2026-09-11) of the op06 gate result found it
# tests the wrong thing: submission.pool_logit_mean gives slab2d a FIXED,
# un-learnable 1/7 (14.3%) weight in the pooled logit, diluting it into
# the existing 6-variant pool rather than testing whether it carries
# INCREMENTAL information. The fair test: add the pooled slab2d logit as
# its OWN third blend feature and let the logistic regression learn its
# coefficient (including ~0, if it has nothing to add) -- this was never
# run and costs nothing (same arrays already on disk). Also prints
# slab2d's own solo AUROC, never printed before (log loss alone can't
# distinguish "no signal" from "real signal, badly calibrated").
slab_pooled = submission.pool_logit_mean(slab2d_arrays)
print(f"slab2d alone (pooled, uncalibrated): "
      f"AUROC={evaluate.auroc_score(y_true, slab_pooled):.4f}  "
      f"log loss={evaluate.log_loss_score(y_true, slab_pooled):.4f}")

X3 = np.column_stack([submission.to_logit(cnn_pooled_6variant),
                       submission.to_logit(slab_pooled),
                       submission.to_logit(baseline_pooled)])
candidate_oof_fair = evaluate.oof_predict(X3, y_true, FOLDS)
result_fair = evaluate.paired_gate("6-variant + slab2d as its OWN blend feature (fair test)",
                                    candidate_oof_fair, current_oof, y_true,
                                    min_effect=0.003, seed=config.RANDOM_STATE)

**What we're looking for:** whether `slab2d` -- a genuinely different
architecture (2D thick-slab CNN, Wenzel et al. 2019) trained under a plain
nested-CV protocol -- adds real value as a 7th ensemble member, using the
corrected paired-bootstrap gate. This is roadmap item 6, the last untried
lever with an actual discrimination (AUROC) mechanism after today's three
recombination-only candidates (notebook 23) all came back null.

**What we found** (run 2026-09-11, after the geometry fix in `config.py`/
`data.load_slab` -- see the diagnosis above op02load and `RESOURCES.md`):
- **Cache build**: 4471.4s (~74.5min) for 1362 volumes -- matches a
  projection from this project's own `load_volume` smoke-test benchmark
  (~1.7s/volume) scaled by the ~1.86x voxel-count increase from
  resampling to 2mm instead of 2.46mm over the full head. One-time cost,
  cached to disk.
- **Degenerate/near-empty rate: 29/1362 (2.1%)** -- above the notebook's
  own 1% flag threshold, up from <1% under the (broken) original 1x12mm-
  voxel geometry. Plausible read: the new geometry is *more precisely
  centered* (+-1mm vs +-6mm) but therefore *less forgiving* of real
  per-subject anatomical variation right at the edge of a now much more
  exactly-pinned 12mm window -- not investigated further (see decision
  below).
- **Fold-0 sanity check**: 24 epochs, inner-val best=0.6725, outer fold
  log loss=0.6791 -- barely different from the pre-fix number (0.6786),
  despite the geometry fix being independently confirmed correct via the
  synthetic-marker diagnostic. The inner-val loss history shows real
  instability in later epochs (climbing to 2.3808 before stopping) --
  `train_one_fold` correctly returns the *best* checkpoint (epoch 14,
  val_loss=0.6725), so this doesn't corrupt the reported score, but flags
  a real LR/stability issue independent of geometry, not investigated
  further (see decision below).
- **Full 5x5 nested-CV**: pooled per-seed log loss 0.6718/0.6745/0.6845/
  0.6766/0.6748, mean=**0.6765** sd=0.0048 -- consistently, tightly just
  above the base-rate baseline (0.688443) across all 5 seeds, nowhere
  near the 3D CNN's 0.4520 or even the single-scalar classical feature's
  ~0.61 (README.md). `slab2d` alone is a genuinely weak model on this
  data at this resolution/thickness.
- **Gate** (`evaluate.paired_gate`, 6-variant vs. 7-variant(+slab2d),
  diluted to a fixed 1/7 weight via `submission.pool_logit_mean`):
  delta=**+0.0013** (worse, not better), 95% paired-bootstrap CI=
  **[+0.0001, +0.0025]** -- entirely positive. AUROC 0.9184 (6-variant)
  vs. 0.9180 (7-variant). **Does NOT clear the gate.**
- **op07fairgate** (the un-diluted, fair version of the same question --
  added after a second Opus review): **slab2d alone (pooled, uncalibrated):
  AUROC=0.6269, log loss=0.6610** -- well above 0.5, confirms real (if
  weak) discriminative signal, not zero-information noise, consistent
  with the crop landing on real striatal tissue for only a fraction of
  subjects rather than for none. **Fair gate**: delta=**-0.0009** (now
  slightly favorable, not unfavorable), 95% CI=**[-0.0034, +0.0017]** --
  crosses zero. **Does not clear, but this is now a genuinely NULL result
  (statistically indistinguishable from zero effect), not the diluted
  gate's reliably-worse verdict.** The two gates disagree in direction
  because they test different things -- the diluted gate measures the
  blend's robustness to being watered down by 1/7 of a weak model; this
  one measures whether slab2d's own AUROC-0.63 signal is incremental
  over what the 6-variant+baseline blend already has, and the honest
  answer is "can't tell, with this geometry."

**Decision (revised after a second Opus review, 2026-09-11): `slab2d` is
NOT adopted, but this is NOT a decisive "this architecture doesn't work
here" finding -- it is an unvalidated design, not a negatively-validated
one.** The first pass through this reflection cell claimed a "correctly-
implemented" negative result; that was wrong, and both `RESOURCES.md` and
`config.py`'s comments have since been corrected. What the second review
actually found, verified against the code (no coding bug -- `crop_or_pad`,
the 6-voxel mean, axis order, and `train_one_fold`'s best-checkpoint logic
are all correct):

- `config.CROP_CENTER_MM` is the **population-median** row of
  `notebooks/01_eda_volumes.ipynb` section 6a's per-volume striatum-
  centroid-offset table, whose z component has **sd ~32.5mm** (raw-axis
  frame, n=119) -- a 12mm-thick window fixed at that one point contains
  real striatal tissue for only ~14-47% of subjects by a normal
  approximation, vs. ~89% for the 3D track's 108mm-thick crop (which
  tolerates the same offset error by sheer thickness, not because its
  geometry is more correct).
- This is why the fold-0 sanity check barely moved after the earlier
  geometry-precision fix (0.6786->0.6791): that fix corrected a ±6mm->
  ±1mm rounding error, roughly 2 orders of magnitude smaller than the
  ~32mm population spread that was never addressed.
- The 2.1% degenerate-crop rate is fully explained, not mysterious:
  `normalize_intensity`/`_warn_if_degenerate` only fires on a LITERALLY
  all-zero crop (verified from the code), so those subjects' slabs missed
  the head entirely -- and the rate *rose* after the "fix" because the
  offset distribution is left-skewed (mean -25.1mm vs. median -15.5mm),
  so the unfixed geometry (which happened to sit nearer -18 to -24mm) was
  accidentally closer to the population's center of mass than the fixed
  one.
- Independent confirmation the crop mostly isn't landing on real striatal
  tissue: a single hand-crafted scalar, `abs_asym` alone, scores 0.5889
  log loss (README.md) -- `slab2d`'s 2.87M-parameter CNN scoring 0.6765
  (worse) is not plausible if it were actually seeing the anatomy.

**What this means going forward**: `notebooks/25_striatum_centroid_ras_frame_audit.ipynb`
redoes the offset measurement correctly (RAS-resampled frame, not raw
axes; all 1362 volumes, not 119) and, more importantly, audits whether
the *production* 3D crop itself fully contains the striatum for every
subject -- a potentially bigger, previously-unidentified lever on the
shipped 0.4185 model than `slab2d` ever was. Per-subject centering for a
retrained `slab2d` (via `features.striatum_mask`) was judged worth
~15-25% odds and ~4h of work by the second review *if* notebook 25 shows
the offset spread is real and large in the correct frame -- not attempted
here; that decision waits on notebook 25's results.

**`checkpoints/slab2d_*.pt` and `slab2d_oof_seed*.npy` stay on disk,
unused in production either way** -- this was local-CV-only work, zero
submission cost, zero risk to the shipped 0.4185 recipe throughout.

**Status of the whole investigation, corrected**: the "0.4185 is final"
conclusion reached earlier today is **not yet shown to be true** -- it
was resting partly on this notebook's now-corrected overclaim. See
`notebooks/25_striatum_centroid_ras_frame_audit.ipynb` for what's next.